# Phase 1 — Legal Language Model Pretraining (CS781 Capstone) — Kaggle, **fixed final version**

1. **Out-of-memory, silently.** Every `train.py` / `eval.py` / `distill.py` process did `json.load()` on all four files (~19 GB of JSON → far more than Kaggle's 31 GB RAM once Python objects are built). The kernel SIGKILLed it (no traceback), `!python` swallowed the exit code, and the notebook carried on. **Fix:** `prepare_data.py` streams the JSON once (ijson), tokenizes in batches and stores `uint16` shards; all runs then use `np.memmap`. Every command now goes through `run()`, which **stops the notebook** on a non-zero exit code.
2. **Tier-1 models were below the 50M floor** (`decoder_only` 512×8 = 40.0M) so `train.py`'s own assertion would have fired next. Configs re-sized (57M / 62M / 57M / 51M).
3. **Off-by-one label bug:** dataset shifted the labels *and* the model shifted again → every model learned to predict token *t+2*. Fixed.
4. **`chunk_recurrent` leaked the future:** every token could cross-attend to memory built from the whole sequence (including the token being predicted). Now causal, with a learned empty-memory slot.
5. **Tokenizer only saw statutes:** `--max_docs` took the first N docs and `train_acts` comes first. Now a random sample from **all four** sources, with all 256 bytes in the alphabet (no `<unk>`).
6. **Prompt ended with `<eos>`** at generation time (tokenizer template) → the model was told the document was over. Tokenizer now adds no specials; `<bos>`/`<eos>` are explicit. `enc_dec` gets the *last* 600 prompt tokens instead of the first.
7. **`torch.load` would have failed** on torch ≥ 2.6 (dataclass config, `weights_only=True`). Configs are stored as dicts.
8. `pip install --upgrade tokenizers` broke `transformers`' pin (visible in the log) → could break BERTScore. No more `--upgrade`.
9. Validation sets differed between the data-mix ablation runs (val was drawn with the *training* mix weights) → not comparable. Val is now a fixed, equal-weight set.
10. bf16 autocast on T4/P100 (no native bf16) → now fp16 + GradScaler there, bf16 on Ampere+. Boolean attention masks (float32 masks crash SDPA under autocast).
11. Tier-2 was 20 000 steps (≈ 650M tokens, ≈ 7 h on a T4) and would never have finished; now sized to the session (with a wall-clock guard `--time_budget_min`).

**Kaggle setup:** add your dataset as an Input · Accelerator **GPU (T4 ×2 or P100)** · **Internet On** (pip + BERTScore's roberta-large download). Everything is written under `/kaggle/working`.

**Workflow:** run once with `QUICK = True` (≈ 30–40 min, exercises every cell on tiny budgets), then set `QUICK = False` and delete `/kaggle/working/artifacts` (or set `FORCE_REBUILD = True`) for the real run (≈ 7–8 h on a T4 — see the settings cell to scale it).

## 0. Settings + dependencies

In [ ]:
QUICK = False          # True  -> tiny budgets, whole notebook in ~30-40 min (smoke test of every cell)
FORCE_REBUILD = False  # True  -> ignore cached tokenizer / tokenized shards under artifacts/

if QUICK:
    S = dict(tok_mb=6,  tok_docs=400,  prep_docs=3000, val_frac=0.05, eval_tokens=100_000,
             t1_tokens=1_000_000, t1_steps=40,  t1_eval=20,  t1_warm=5,
             abl_tokens=1_000_000, abl_steps=30, abl_eval=15, abl_warm=5,
             t2_tokens=2_000_000, t2_steps=60,  t2_eval=20,  t2_warm=5, t2_patience=0, t2_budget=0,
             n_eval_t1=4, n_eval=4, dist_steps=30, dist_tokens=1_000_000, dist_eval=15)
else:
    S = dict(tok_mb=60, tok_docs=0,    prep_docs=0,    val_frac=0.02, eval_tokens=400_000,
             t1_tokens=25_000_000, t1_steps=1500, t1_eval=150, t1_warm=100,
             abl_tokens=16_000_000, abl_steps=1000, abl_eval=250, abl_warm=60,
             t2_tokens=200_000_000, t2_steps=5000, t2_eval=500, t2_warm=300, t2_patience=6, t2_budget=180,
             n_eval_t1=40, n_eval=60, dist_steps=2500, dist_tokens=20_000_000, dist_eval=500)
PREP_CAP = 250_000_000      # max tokens kept per source when tokenizing (acts is smaller and is kept whole)
print("QUICK" if QUICK else "FULL", "settings:", S)

In [ ]:
# Internet must be On. NOTE: no "--upgrade tokenizers" (that broke transformers' version pin in the last run).
!pip install -q ijson bert-score rouge-score sacrebleu

In [ ]:
import importlib, os, subprocess
for m in ["torch", "tokenizers", "transformers", "ijson", "bert_score", "rouge_score", "sacrebleu", "pandas"]:
    try:
        mod = importlib.import_module(m); print(f"{m:14s} {getattr(mod, '__version__', 'ok')}")
    except Exception as e:
        print(f"{m:14s} FAILED: {type(e).__name__}: {e}")
import ijson, torch
print("ijson backend:", ijson.backend, "(want yajl2_c; 'python' is ~20x slower)")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB, compute capability {p.major}.{p.minor} "
          f"-> autocast {'bf16' if p.major >= 8 else 'fp16'}")
else:
    print("WARNING: no GPU visible -- enable an Accelerator in the notebook settings")
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)

## 1. Locate the data (§7.2)

Kaggle extracts an uploaded zip dataset under `/kaggle/input/...`; the nesting depends on the zip, so we search by file name.

In [ ]:
import os

KAGGLE_INPUT = "/kaggle/input"
print("Available input datasets:", os.listdir(KAGGLE_INPUT))
REQUIRED_SOURCES = ["train_acts", "train_high_courts", "train_supreme_court", "train_tribunals"]

def find_file(base, filename):
    for root, dirs, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

found, missing = {}, []
for name in REQUIRED_SOURCES:
    path = find_file(KAGGLE_INPUT, f"{name}.json")
    if path:
        found[name] = path
        print(f"OK       {name}.json  ->  {path}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        missing.append(name); print(f"MISSING  {name}.json")
assert not missing, f"Missing sources: {missing}"

test_path = (find_file(KAGGLE_INPUT, "test.json") or find_file(KAGGLE_INPUT, "Phase1- Test.json")
             or find_file(KAGGLE_INPUT, "Phase1-Test.json"))
assert test_path is not None, "Could not find the test .json under /kaggle/input"
train_dirs = {os.path.dirname(p) for p in found.values()}
assert len(train_dirs) == 1, f"Train files spread across multiple folders: {train_dirs}"
TRAIN_DIR, TEST_FILE = train_dirs.pop(), test_path
print("\nTRAIN_DIR:", TRAIN_DIR, "\nTEST_FILE:", TEST_FILE)

# /kaggle/input is read-only: everything we write goes under /kaggle/working
os.makedirs("/kaggle/working/artifacts", exist_ok=True)
os.chdir("/kaggle/working")
DATA_DIR = "artifacts/data"
print("Working directory:", os.getcwd())

## 2. Write project files

`config.py` · `models.py` (4 architectures, chunk-recurrent made causal) · `prepare_data.py` (**streaming** tokenize → uint16 shards) · `data_utils.py` (memmap datasets) · `tokenizer_train.py` · `train.py` · `generate.py` · `eval.py` · `distill.py`.

In [ ]:
%%writefile config.py
"""
Central configuration for Phase-1 Legal LM Pretraining.
Edit paths / hyperparams here rather than scattering magic numbers in scripts.
"""

from dataclasses import dataclass, field
from typing import Optional


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
@dataclass
class PathConfig:
    train_dir: str = "Phase_1_Pretraining/Train"
    test_file: str = "Phase_1_Pretraining/Test/Phase1- Test.json"
    data_dir: str = "artifacts/data"          # tokenized shards written by prepare_data.py
    tokenizer_out: str = "artifacts/tokenizer.json"
    checkpoint_dir: str = "artifacts/checkpoints"
    prediction_out: str = "artifacts/prediction.json"
    submission_zip: str = "artifacts/submission.zip"


# ---------------------------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------------------------
@dataclass
class TokenizerConfig:
    vocab_size: int = 32000
    min_frequency: int = 2
    special_tokens: tuple = ("<pad>", "<bos>", "<eos>", "<unk>")


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
@dataclass
class DataConfig:
    context_length: int = 512          # sequence length used for LM training
    val_fraction: float = 0.02         # held-out fraction, split by document
    # relative sampling weights per source; tune via ablation (see §7.4)
    source_weights: dict = field(default_factory=lambda: {
        "train_acts": 1.0,
        "train_high_courts": 1.5,
        "train_supreme_court": 1.5,
        "train_tribunals": 1.0,
    })
    max_train_tokens: Optional[int] = 100_000_000  # cap for Tier-1 cheap search; raise for Tier-2
    seed: int = 42


# ---------------------------------------------------------------------------
# Model — shared knobs, kept under ~300M params for every architecture
# ---------------------------------------------------------------------------
@dataclass
class ModelConfig:
    architecture: str = "decoder_only"   # decoder_only | enc_dec | efficient | chunk_recurrent
    vocab_size: int = 32000
    hidden_size: int = 768
    num_layers: int = 12
    num_heads: int = 12
    num_kv_heads: int = 4          # for GQA in decoder_only / efficient
    ffn_multiplier: float = 2.75   # SwiGLU effective expansion (~4x dense equivalent)
    max_position_embeddings: int = 1024
    dropout: float = 0.1
    # efficient-architecture specific
    local_window: int = 128
    global_every: int = 4          # every Nth layer is full attention
    # chunk-recurrent specific
    chunk_size: int = 128
    memory_size: int = 32          # summary vector width multiplier


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
@dataclass
class TrainConfig:
    batch_size: int = 16
    grad_accum_steps: int = 4
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_steps: int = 500
    max_steps: int = 20000
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_every: int = 500
    save_every: int = 1000
    log_every: int = 50
    mixed_precision: str = "bf16"   # "bf16" | "fp16" | "no"
    compile_model: bool = True


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
@dataclass
class GenConfig:
    max_new_words: int = 128
    temperature: float = 0.8
    top_p: float = 0.9
    top_k: int = 50
    repetition_penalty: float = 1.2
    no_repeat_ngram_size: int = 4

In [ ]:
%%writefile models.py
"""
Four substantially different architectures for Phase-1 legal LM pretraining.
All kept in the 50M-300M parameter range by adjusting hidden_size / num_layers
in config.py.

    1. DecoderOnlyLM        - RoPE + GQA + SwiGLU pre-norm transformer (baseline)
    2. Seq2SeqLM            - Encoder-decoder transformer (natural fit for
                               "512-word prompt -> 128-word continuation")
    3. EfficientLocalGlobalLM - local sliding-window attention with periodic
                               full-attention layers (cheap long-range mixing)
    4. ChunkRecurrentLM     - novel: encodes the prompt in fixed-size chunks,
                               compresses each chunk into a memory vector via
                               a small recurrent update, decoder cross-attends
                               to accumulated memory + local window
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================================
# Shared primitives
# ============================================================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=2.75, dropout=0.1):
        super().__init__()
        hidden = int(dim * mult)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


def rotary_freqs(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cat([freqs, freqs], dim=-1)  # (seq_len, head_dim)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, freqs):
    cos, sin = freqs.cos(), freqs.sin()
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


class GQAAttention(nn.Module):
    """Grouped-query causal self-attention with RoPE. Supports an optional
    additive attn_mask (for local-window / block patterns)."""

    def __init__(self, dim, num_heads, num_kv_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)
        self.drop = dropout

    def forward(self, x, freqs, attn_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, freqs)

        rep = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=attn_mask is None,
            dropout_p=self.drop if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)


def build_local_window_mask(T, window, device):
    """Boolean mask: True = attend. Causal + only last `window` tokens.
    (Boolean instead of an additive float mask: SDPA rejects a float32 mask
    when q/k/v are bf16/fp16 under autocast.)"""
    idx = torch.arange(T, device=device)
    causal = idx[None, :] <= idx[:, None]
    local = (idx[:, None] - idx[None, :]) < window
    return (causal & local)[None, None, :, :]


def build_causal_mask(T, device):
    idx = torch.arange(T, device=device)
    return (idx[None, :] <= idx[:, None])[None, None, :, :]


# ============================================================================
# 1. Decoder-only Transformer (baseline: RoPE + GQA + SwiGLU, pre-norm)
# ============================================================================
class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, attn_mask=None):
        x = x + self.attn(self.norm1(x), freqs, attn_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        for block in self.blocks:
            x = block(x, freqs)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.9,
                 top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=4,
                 eos_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            T = input_ids.size(1)
            if T > self.cfg.max_position_embeddings:
                input_ids = input_ids[:, -self.cfg.max_position_embeddings:]
            logits, _ = self.forward(input_ids)
            next_logits = logits[:, -1, :].float() / max(temperature, 1e-5)
            next_logits = _apply_repetition_penalty(next_logits, input_ids, repetition_penalty)
            next_logits = _block_repeat_ngrams(next_logits, input_ids, no_repeat_ngram_size)
            next_logits = _top_k_top_p_filter(next_logits, top_k, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if eos_token_id is not None and (next_token == eos_token_id).all():
                break
        return input_ids


def _apply_repetition_penalty(logits, generated, penalty):
    if penalty == 1.0:
        return logits
    for b in range(generated.size(0)):
        seen = torch.unique(generated[b])
        vals = logits[b, seen]
        logits[b, seen] = torch.where(vals < 0, vals * penalty, vals / penalty)
    return logits


def _block_repeat_ngrams(logits, generated, n):
    if n <= 0 or generated.size(1) < n:
        return logits
    B, T = generated.shape
    for b in range(B):
        seq = generated[b].tolist()
        if len(seq) < n - 1:
            continue
        prefix = tuple(seq[-(n - 1):])
        banned = set()
        for i in range(len(seq) - n + 1):
            if tuple(seq[i:i + n - 1]) == prefix:
                banned.add(seq[i + n - 1])
        for tok in banned:
            logits[b, tok] = float("-inf")
    return logits


def _top_k_top_p_filter(logits, top_k, top_p):
    if top_k > 0:
        top_k = min(top_k, logits.size(-1))
        kth_vals = torch.topk(logits, top_k, dim=-1).values[..., -1, None]
        logits = torch.where(logits < kth_vals, torch.full_like(logits, float("-inf")), logits)
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = F.softmax(sorted_logits, dim=-1)
        cum_probs = torch.cumsum(probs, dim=-1)
        remove = cum_probs > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        sorted_logits[remove] = float("-inf")
        logits = torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    return logits


# ============================================================================
# 2. Encoder-Decoder Transformer (natural fit: prompt -> continuation)
# ============================================================================
class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                           dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderLayerCross(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                 dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, memory, self_mask, mem_key_padding_mask=None):
        h = self.norm1(x)
        sa, _ = self.self_attn(h, h, h, attn_mask=self_mask, need_weights=False)
        x = x + sa
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory,
                                 key_padding_mask=mem_key_padding_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class Seq2SeqLM(nn.Module):
    """Encoder reads the 512-word prompt; decoder autoregressively produces
    the continuation. Uses learned positional embeddings (simplest to reason
    about across two separate sequences)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        half_layers = max(1, cfg.num_layers // 2)
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_emb = nn.Embedding(cfg.max_position_embeddings, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(half_layers)])
        self.decoder = nn.ModuleList([DecoderLayerCross(cfg) for _ in range(half_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _embed(self, ids):
        T = ids.size(1)
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        return self.drop(self.tok_emb(ids) + self.pos_emb(pos))

    def encode(self, src_ids, src_padding_mask=None):
        x = self._embed(src_ids)
        for layer in self.encoder:
            x = layer(x, key_padding_mask=src_padding_mask)
        return x

    def forward(self, src_ids, tgt_ids, labels=None, src_padding_mask=None):
        memory = self.encode(src_ids, src_padding_mask)
        x = self._embed(tgt_ids)
        T = tgt_ids.size(1)
        causal = torch.triu(torch.full((T, T), float("-inf"), device=tgt_ids.device), diagonal=1)
        for layer in self.decoder:
            x = layer(x, memory, causal, mem_key_padding_mask=src_padding_mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, eos_id, max_new_tokens, temperature=0.8,
                 top_p=0.9, top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=4,
                 src_padding_mask=None):
        self.eval()
        memory = self.encode(src_ids, src_padding_mask)
        tgt = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self._embed(tgt)
            T = tgt.size(1)
            causal = torch.triu(torch.full((T, T), float("-inf"), device=tgt.device), diagonal=1)
            h = x
            for layer in self.decoder:
                h = layer(h, memory, causal, mem_key_padding_mask=src_padding_mask)
            h = self.norm_f(h)
            next_logits = self.lm_head(h)[:, -1, :].float() / max(temperature, 1e-5)
            next_logits = _apply_repetition_penalty(next_logits, tgt, repetition_penalty)
            next_logits = _block_repeat_ngrams(next_logits, tgt, no_repeat_ngram_size)
            next_logits = _top_k_top_p_filter(next_logits, top_k, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tgt = torch.cat([tgt, next_token], dim=1)
            if (next_token == eos_id).all():
                break
        return tgt


# ============================================================================
# 3. Efficient / lightweight Transformer: local sliding-window attention with
#    periodic full-attention layers for global mixing.
# ============================================================================
class EfficientLocalGlobalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        local_mask = build_local_window_mask(T, self.cfg.local_window, input_ids.device)
        full_mask = build_causal_mask(T, input_ids.device)
        for i, block in enumerate(self.blocks):
            is_global = (i + 1) % self.cfg.global_every == 0
            mask = full_mask if is_global else local_mask
            x = block(x, freqs, attn_mask=mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, **kw):
        self.eval()
        for _ in range(max_new_tokens):
            T = input_ids.size(1)
            if T > self.cfg.max_position_embeddings:
                input_ids = input_ids[:, -self.cfg.max_position_embeddings:]
            logits, _ = self.forward(input_ids)
            next_logits = logits[:, -1, :].float() / max(kw.get("temperature", 0.8), 1e-5)
            next_logits = _apply_repetition_penalty(next_logits, input_ids, kw.get("repetition_penalty", 1.2))
            next_logits = _block_repeat_ngrams(next_logits, input_ids, kw.get("no_repeat_ngram_size", 4))
            next_logits = _top_k_top_p_filter(next_logits, kw.get("top_k", 50), kw.get("top_p", 0.9))
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if kw.get("eos_token_id") is not None and (next_token == kw["eos_token_id"]).all():
                break
        return input_ids


# ============================================================================
# 4. Chunk-Recurrent Hybrid (novel architecture):
#    Prompt is split into fixed-size chunks. Each chunk is summarized into a
#    memory vector via a GRU-style recurrent update (memory carried across
#    chunks, cheap O(T) alternative to full attention over very long
#    contexts). The decoder then does local causal attention over recent
#    tokens PLUS cross-attention to the accumulated memory bank -- letting it
#    reference facts from far earlier in a long legal prompt (e.g. an early
#    citation or defined term) without full quadratic attention.
# ============================================================================
class ChunkEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.chunk_size = cfg.chunk_size
        self.hidden = cfg.hidden_size
        self.intra_chunk_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                        dropout=cfg.dropout, batch_first=True)
        self.norm = RMSNorm(cfg.hidden_size)
        self.summarize = nn.Linear(cfg.hidden_size, cfg.hidden_size)
        self.recurrent_update = nn.GRUCell(cfg.hidden_size, cfg.hidden_size)

    def forward(self, x):
        """x: (B, T, H) prompt embeddings. Returns memory bank (B, num_chunks, H)."""
        B, T, H = x.shape
        n_chunks = math.ceil(T / self.chunk_size)
        pad = n_chunks * self.chunk_size - T
        if pad > 0:
            x = F.pad(x, (0, 0, 0, pad))
        chunks = x.view(B, n_chunks, self.chunk_size, H)

        state = torch.zeros(B, H, device=x.device, dtype=x.dtype)
        memories = []
        for c in range(n_chunks):
            seg = chunks[:, c]                      # (B, chunk_size, H)
            h = self.norm(seg)
            attn_out, _ = self.intra_chunk_attn(h, h, h, need_weights=False)
            seg = seg + attn_out
            pooled = self.summarize(seg.mean(dim=1))  # (B, H) mean-pool the chunk
            state = self.recurrent_update(pooled, state)
            memories.append(state)
        return torch.stack(memories, dim=1)  # (B, n_chunks, H)


class ChunkRecurrentBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                 dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, memory, mem_mask=None):
        x = x + self.self_attn(self.norm1(x), freqs)  # causal self-attn
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory, attn_mask=mem_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class ChunkRecurrentLM(nn.Module):
    """Decoder-only interface (matches the pure-continuation task) but with
    chunk-recurrent memory injected via cross-attention at every layer."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.chunk_encoder = ChunkEncoder(cfg)
        self.blocks = nn.ModuleList([ChunkRecurrentBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        # learned "empty memory" slot so tokens in the first chunk have something to attend to
        self.null_mem = nn.Parameter(torch.zeros(1, 1, cfg.hidden_size))
        self.apply(self._init_weights)
        nn.init.normal_(self.null_mem, mean=0.0, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        memory = self.chunk_encoder(x.detach())            # (B, n_chunks, H): summary of chunk c
        n_chunks = memory.size(1)
        memory = torch.cat([self.null_mem.expand(B, -1, -1).to(memory.dtype), memory], dim=1)  # slot 0 = empty
        # CAUSALITY FIX: a token in chunk c may only read slot 0 and the summaries of chunks 0..c-1
        # (slots 1..c). The old code let every token read every chunk, including the chunk that
        # contains the very token it is asked to predict, so its val loss was artificially low.
        chunk_id = torch.arange(T, device=input_ids.device) // self.cfg.chunk_size
        slot = torch.arange(n_chunks + 1, device=input_ids.device)
        mem_mask = slot[None, :] > chunk_id[:, None]        # True = NOT allowed
        for block in self.blocks:
            x = block(x, freqs, memory, mem_mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, **kw):
        self.eval()
        for _ in range(max_new_tokens):
            T = input_ids.size(1)
            if T > self.cfg.max_position_embeddings:
                input_ids = input_ids[:, -self.cfg.max_position_embeddings:]
            logits, _ = self.forward(input_ids)
            next_logits = logits[:, -1, :].float() / max(kw.get("temperature", 0.8), 1e-5)
            next_logits = _apply_repetition_penalty(next_logits, input_ids, kw.get("repetition_penalty", 1.2))
            next_logits = _block_repeat_ngrams(next_logits, input_ids, kw.get("no_repeat_ngram_size", 4))
            next_logits = _top_k_top_p_filter(next_logits, kw.get("top_k", 50), kw.get("top_p", 0.9))
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if kw.get("eos_token_id") is not None and (next_token == kw["eos_token_id"]).all():
                break
        return input_ids


# ============================================================================
# Factory + param counting
# ============================================================================
def build_model(cfg):
    arch = cfg.architecture
    if arch == "decoder_only":
        return DecoderOnlyLM(cfg)
    if arch == "enc_dec":
        return Seq2SeqLM(cfg)
    if arch == "efficient":
        return EfficientLocalGlobalLM(cfg)
    if arch == "chunk_recurrent":
        return ChunkRecurrentLM(cfg)
    raise ValueError(f"Unknown architecture: {arch}")


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================================
# Checkpoint / AMP helpers shared by train / eval / generate / distill
# ============================================================================
def get_amp_dtype():
    """bf16 on Ampere+ GPUs, fp16 on T4/P100 (no native bf16), None on CPU."""
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def load_checkpoint(path, device):
    """Load a checkpoint written by train.py / distill.py. Configs are stored as plain dicts,
    so this works on torch>=2.6 where torch.load defaults to weights_only=True and cannot
    unpickle a dataclass (the old torch.load(path) calls would have failed)."""
    from config import ModelConfig
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    if isinstance(cfg, dict):
        cfg = ModelConfig(**cfg)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, cfg, ckpt

In [ ]:
%%writefile prepare_data.py
"""
One-pass, memory-safe preparation of the raw JSON corpus (no torch needed).

WHY: the raw train files are ~19 GB of JSON. The old pipeline called json.load() on all four files
in EVERY training / eval / distill process. That needs far more than Kaggle's 31 GB of RAM, so each
process was silently OOM-killed ("Loading corpus..." and then nothing) and no checkpoint was ever
written. This script streams each file record-by-record (ijson), tokenizes in batches and writes
compact uint16 token files. Everything downstream reads them through np.memmap.

Outputs in --out_dir:
    <source>.{train,val}.bin           uint16 tokens, each doc = <bos> ... <eos>
    <source>.{train,val}.offsets.npy   uint64 doc start offsets (len n_docs+1)
    val_pairs.json                     held-out (512-word prompt, next-128-word reference) pairs
    meta.json                          vocab / special ids / per-source statistics

Split is by whole document (val docs never appear in train).
"""
import argparse
import json
import os
import random
import time

import numpy as np

SOURCES = ["train_acts", "train_high_courts", "train_supreme_court", "train_tribunals"]
BYTES_PER_TOKEN = 5.0   # rough JSON-bytes per token, used only to choose a sampling rate


def _text_of(rec):
    t = rec.get("text") if isinstance(rec, dict) else rec
    return t if isinstance(t, str) and t.strip() else None


def _first_char(path):
    with open(path, "rb") as f:
        while True:
            c = f.read(1)
            if not c:
                return ""
            if not c.isspace():
                return c.decode("utf-8", "ignore")


def iter_docs(path):
    """Yield the text of every record in a (huge) JSON file without loading the file.
    Handles both  {"id": {"text": ...}, ...}  and  [{"text": ...}, ...]."""
    import ijson
    first = _first_char(path)
    with open(path, "rb") as f:
        if first == "{":
            for _, rec in ijson.kvitems(f, "", use_float=True):
                t = _text_of(rec)
                if t:
                    yield t
        else:
            for rec in ijson.items(f, "item", use_float=True):
                t = _text_of(rec)
                if t:
                    yield t


def main():
    from tokenizers import Tokenizer
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_dir", required=True)
    ap.add_argument("--tokenizer_path", default="artifacts/tokenizer.json")
    ap.add_argument("--out_dir", default="artifacts/data")
    ap.add_argument("--max_tokens_per_source", type=int, default=250_000_000)
    ap.add_argument("--val_fraction", type=float, default=0.02)
    ap.add_argument("--val_pairs_per_source", type=int, default=400)
    ap.add_argument("--stop_after_docs", type=int, default=0,
                    help="debug/QUICK mode: scan only the first N docs of each source (0 = all)")
    ap.add_argument("--max_doc_chars", type=int, default=1_000_000)
    ap.add_argument("--batch_docs", type=int, default=256)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    tok = Tokenizer.from_file(args.tokenizer_path)
    vocab = tok.get_vocab_size()
    assert vocab < 65536, "uint16 token storage needs vocab_size < 65536"
    pad, bos, eos = (tok.token_to_id(t) for t in ("<pad>", "<bos>", "<eos>"))
    assert None not in (pad, bos, eos)

    meta = {"vocab_size": vocab, "pad_id": pad, "bos_id": bos, "eos_id": eos, "sources": {}}
    val_pairs = []

    for name in SOURCES:
        path = os.path.join(args.train_dir, f"{name}.json")
        if not os.path.exists(path):
            print(f"[warn] missing {path}, skipping")
            continue
        fsize = os.path.getsize(path)
        keep_p = min(1.0, args.max_tokens_per_source * BYTES_PER_TOKEN / fsize)
        rng = random.Random(f"{args.seed}-{name}")
        fh = {s: open(os.path.join(args.out_dir, f"{name}.{s}.bin"), "wb") for s in ("train", "val")}
        lens = {"train": [], "val": []}
        ntok = {"train": 0, "val": 0}
        st = {"scanned": 0, "kept": 0, "pairs": 0}
        buf = []
        t0 = time.time()
        last_print = t0

        def flush():
            if not buf:
                return
            encs = tok.encode_batch([t for _, t in buf], add_special_tokens=False)
            for (split, _), enc in zip(buf, encs):
                arr = np.asarray([bos] + enc.ids + [eos], dtype=np.uint16)
                arr.tofile(fh[split])
                lens[split].append(len(arr))
                ntok[split] += len(arr)
            buf.clear()

        print(f"[{name}] {fsize/1e9:.2f} GB, keep probability {keep_p:.3f}", flush=True)
        for text in iter_docs(path):
            st["scanned"] += 1
            if args.stop_after_docs and st["scanned"] > args.stop_after_docs:
                break
            if keep_p < 1.0 and rng.random() > keep_p:
                continue
            text = text[: args.max_doc_chars]
            is_val = rng.random() < args.val_fraction
            st["kept"] += 1
            if is_val and st["pairs"] < args.val_pairs_per_source:
                words = text.split()
                if len(words) >= 512 + 16:
                    val_pairs.append({"source": name,
                                      "prompt": " ".join(words[:512]),
                                      "reference": " ".join(words[512:512 + 128])})
                    st["pairs"] += 1
            buf.append(("val" if is_val else "train", text))
            if len(buf) >= args.batch_docs:
                flush()
            if ntok["train"] > 1.5 * args.max_tokens_per_source:   # safety cap
                print(f"[{name}] hit 1.5x token cap, stopping early")
                break
            if time.time() - last_print > 60:
                last_print = time.time()
                print(f"[{name}] scanned {st['scanned']:,} docs | kept {st['kept']:,} | "
                      f"train tok {ntok['train']/1e6:.1f}M | {time.time()-t0:.0f}s", flush=True)
        flush()
        for f in fh.values():
            f.close()
        for s in ("train", "val"):
            offs = np.concatenate([[0], np.cumsum(lens[s], dtype=np.uint64)]).astype(np.uint64)
            np.save(os.path.join(args.out_dir, f"{name}.{s}.offsets.npy"), offs)
        meta["sources"][name] = {
            "keep_p": keep_p, "docs_scanned": st["scanned"], "docs_kept": st["kept"],
            "train_docs": len(lens["train"]), "val_docs": len(lens["val"]),
            "train_tokens": ntok["train"], "val_tokens": ntok["val"], "val_pairs": st["pairs"],
        }
        print(f"[{name}] done in {time.time()-t0:.0f}s: {meta['sources'][name]}", flush=True)
        if ntok["train"] == 0:
            raise RuntimeError(f"{name}: no training tokens were produced -- check the JSON structure")
        if ntok["val"] == 0:
            print(f"[warn] {name}: 0 validation tokens (source has too few docs for val_fraction)")

    with open(os.path.join(args.out_dir, "val_pairs.json"), "w", encoding="utf-8") as f:
        json.dump(val_pairs, f, ensure_ascii=False)
    with open(os.path.join(args.out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    tot = sum(v["train_tokens"] for v in meta["sources"].values())
    print(f"\nTOTAL train tokens: {tot/1e6:.1f}M | val pairs: {len(val_pairs)} | wrote {args.out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile data_utils.py
"""
Datasets over the pre-tokenized shards written by prepare_data.py (np.memmap, tiny RAM footprint).

  - sample_blocks + PackedLMDataset : fixed-length blocks for decoder_only / efficient / chunk_recurrent
  - WindowSeq2SeqDataset            : (600-token source, ~222-token target) windows for enc_dec

Label convention (BUG FIX): the models shift internally (logits[:, :-1] vs labels[:, 1:]), so labels
must be the SAME tokens as input_ids. The old dataset pre-shifted the labels as well, which trained
every model to predict the token AFTER next.
"""
import json
import os

import numpy as np
import torch
from torch.utils.data import Dataset

SOURCES = ["train_acts", "train_high_courts", "train_supreme_court", "train_tribunals"]
SRC_LEN = 600     # enc_dec source tokens (~490 words, tail of a 512-word prompt fits)
TGT_LEN = 224     # enc_dec target tokens incl. <bos>/<eos> (~175 words, we trim to 128 words)


class TokenStore:
    def __init__(self, data_dir):
        with open(os.path.join(data_dir, "meta.json")) as f:
            self.meta = json.load(f)
        self.vocab_size = self.meta["vocab_size"]
        self.pad_id, self.bos_id, self.eos_id = self.meta["pad_id"], self.meta["bos_id"], self.meta["eos_id"]
        self.tokens, self.offsets = {}, {}
        for name in self.meta["sources"]:
            for split in ("train", "val"):
                p = os.path.join(data_dir, f"{name}.{split}.bin")
                if os.path.exists(p) and os.path.getsize(p) > 0:
                    self.tokens[(name, split)] = np.memmap(p, dtype=np.uint16, mode="r")
                    self.offsets[(name, split)] = np.load(os.path.join(data_dir, f"{name}.{split}.offsets.npy"))

    def names(self, split):
        return [n for n in SOURCES if (n, split) in self.tokens]


def _weights(names, source_weights):
    w = np.array([1.0 if source_weights is None else source_weights.get(n, 1.0) for n in names], dtype=np.float64)
    return w / w.sum()


def sample_blocks(store, split, source_weights, ctx, max_tokens, seed=42):
    """(n_blocks, ctx) uint16 array. Each source contributes max_tokens * weight tokens, taken as random
    non-overlapping ctx-blocks (sampled with replacement of epochs if a source is smaller than its share).
    source_weights=None -> equal weights (used for validation so val sets are comparable across mixes)."""
    rng = np.random.default_rng(seed)
    names = [n for n in store.names(split) if len(store.tokens[(n, split)]) >= 2 * ctx]
    assert names, f"no {split} data with >= {2*ctx} tokens"
    w = _weights(names, source_weights)
    total_blocks = max(1, max_tokens // ctx)
    counts = np.floor(w * total_blocks).astype(int)
    counts[np.argmax(w)] += total_blocks - counts.sum()
    out = np.empty((int(counts.sum()), ctx), dtype=np.uint16)
    pos = 0
    for n, c in zip(names, counts):
        if c == 0:
            continue
        arr = store.tokens[(n, split)]
        n_avail = len(arr) // ctx
        if c <= n_avail:
            idx = rng.choice(n_avail, size=c, replace=False)
        else:
            reps = -(-c // n_avail)
            idx = np.concatenate([rng.permutation(n_avail) for _ in range(reps)])[:c]
        idx = np.sort(idx)                      # sequential-ish disk reads
        for i in idx:
            out[pos] = arr[i * ctx:(i + 1) * ctx]
            pos += 1
    rng.shuffle(out)                            # shuffles rows in place
    return out


class PackedLMDataset(Dataset):
    def __init__(self, blocks):
        self.blocks = blocks

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.blocks[idx].astype(np.int64))
        return {"input_ids": x, "labels": x.clone()}


class WindowSeq2SeqDataset(Dataset):
    """Random (source, target) windows inside documents: source = SRC_LEN consecutive tokens, target =
    <bos> + next (TGT_LEN-2) tokens + <eos>. Half the windows start at the document start (matches the
    test format: prompt = beginning of a document), half at a random offset."""

    def __init__(self, store, split, source_weights, n_examples, seed=42):
        rng = np.random.default_rng(seed)
        self.store, self.split = store, split
        need = SRC_LEN + TGT_LEN - 2
        names, elig = [], {}
        for n in store.names(split):
            offs = store.offsets[(n, split)].astype(np.int64)
            lens = np.diff(offs) - 2                       # content tokens (without <bos>/<eos>)
            e = np.nonzero(lens >= need)[0]
            if len(e):
                names.append(n); elig[n] = (offs, lens, e)
        assert names, "no documents long enough for enc_dec windows"
        w = _weights(names, source_weights)
        self.examples = []                                  # (source, absolute start of source window)
        picks = rng.choice(len(names), size=n_examples, p=w)
        for pi in picks:
            n = names[pi]
            offs, lens, e = elig[n]
            d = e[rng.integers(len(e))]
            slack = int(lens[d] - need)
            start = 0 if rng.random() < 0.5 else int(rng.integers(0, slack + 1))
            self.examples.append((n, int(offs[d]) + 1 + start))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        n, base = self.examples[i]
        arr = self.store.tokens[(n, self.split)]
        src = np.asarray(arr[base:base + SRC_LEN], dtype=np.int64)
        tgt_body = np.asarray(arr[base + SRC_LEN: base + SRC_LEN + TGT_LEN - 2], dtype=np.int64)
        tgt = np.concatenate([[self.store.bos_id], tgt_body, [self.store.eos_id]])
        return {"src_ids": torch.from_numpy(src), "tgt_ids": torch.from_numpy(tgt)}


def collate_lm(batch):
    return {"input_ids": torch.stack([b["input_ids"] for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch])}


def collate_seq2seq(batch, pad_id=0):
    max_src = max(b["src_ids"].size(0) for b in batch)
    max_tgt = max(b["tgt_ids"].size(0) for b in batch)
    src = torch.full((len(batch), max_src), pad_id, dtype=torch.long)
    tgt = torch.full((len(batch), max_tgt), pad_id, dtype=torch.long)
    src_pad_mask = torch.ones((len(batch), max_src), dtype=torch.bool)      # True = padding
    for i, b in enumerate(batch):
        s, t = b["src_ids"], b["tgt_ids"]
        src[i, : s.size(0)] = s
        tgt[i, : t.size(0)] = t
        src_pad_mask[i, : s.size(0)] = False
    labels = tgt.clone()
    labels[tgt == pad_id] = -100
    return {"src_ids": src, "tgt_ids": tgt, "labels": labels, "src_padding_mask": src_pad_mask}

In [ ]:
%%writefile tokenizer_train.py
"""
Train a domain-specific byte-level BPE tokenizer on ALL FOUR corpus sources.

Old bug: the tokenizer was trained on the first N docs of the concatenated stream, and SOURCES starts
with train_acts, so the "32k full-corpus tokenizer" had only ever seen statutes -- never a judgment.
Now we stream every file once (ijson, constant memory), keep a random ~--target_mb_per_source MB
sample from each, cache it as jsonl, and train every vocab size from that cache.

No post-processor: tokenizer.encode() adds NO special tokens. <bos>/<eos> are added explicitly by
prepare_data.py (training) and generate.py (inference). (The old <bos> $A <eos> template put an <eos>
after every prompt at generation time, so the model was asked to continue after "end of document".)

Usage:
    python tokenizer_train.py --train_dir DIR --out artifacts/tokenizer.json --vocab_size 32000
"""
import argparse
import json
import os
import random

from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

from prepare_data import SOURCES, iter_docs


def ensure_sample(train_dir, cache_dir, target_mb_per_source=60, stop_after_docs=0,
                  max_doc_chars=100_000, eval_frac=0.05, seed=42):
    os.makedirs(cache_dir, exist_ok=True)
    tr_path = os.path.join(cache_dir, "tok_train.jsonl")
    ev_path = os.path.join(cache_dir, "tok_eval.jsonl")
    if os.path.exists(tr_path) and os.path.exists(ev_path):
        return tr_path, ev_path
    with open(tr_path + ".tmp", "w", encoding="utf-8") as ftr, open(ev_path + ".tmp", "w", encoding="utf-8") as fev:
        for name in SOURCES:
            path = os.path.join(train_dir, f"{name}.json")
            if not os.path.exists(path):
                print(f"[warn] missing {path}")
                continue
            # debug/QUICK mode (stop_after_docs) reads only the file head, so keep every doc there
            p = 1.0 if stop_after_docs else min(1.0, target_mb_per_source * 1e6 / os.path.getsize(path))
            rng = random.Random(f"{seed}-tok-{name}")
            n = got = 0
            for text in iter_docs(path):
                n += 1
                if stop_after_docs and n > stop_after_docs:
                    break
                if p < 1.0 and rng.random() > p:
                    continue
                rec = json.dumps({"s": name, "t": text[:max_doc_chars]}, ensure_ascii=False)
                (fev if rng.random() < eval_frac else ftr).write(rec + "\n")
                got += 1
            print(f"[{name}] sampled {got:,} of {n:,} docs (p={p:.3f})", flush=True)
    os.replace(tr_path + ".tmp", tr_path)
    os.replace(ev_path + ".tmp", ev_path)
    return tr_path, ev_path


def _texts(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)["t"]


def train_tokenizer(train_jsonl, out, vocab_size, min_frequency=2):
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=min_frequency,
        special_tokens=["<pad>", "<bos>", "<eos>", "<unk>"],       # ids 0,1,2,3
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),      # all 256 bytes -> nothing maps to <unk>
        show_progress=False,
    )
    tokenizer.train_from_iterator(_texts(train_jsonl), trainer=trainer)
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    tokenizer.save(out)
    print(f"Saved tokenizer to {out} (vocab_size={tokenizer.get_vocab_size()})")
    return tokenizer


def report(tokenizer_path, eval_jsonl, label=None):
    """tokens/word on held-out sample docs (never seen by the tokenizer trainer), per source."""
    tok = Tokenizer.from_file(tokenizer_path)
    per = {}
    with open(eval_jsonl, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            w, t = per.setdefault(r["s"], [0, 0])
            per[r["s"]][0] += len(r["t"].split())
            per[r["s"]][1] += len(tok.encode(r["t"], add_special_tokens=False).ids)
    tw = sum(v[0] for v in per.values()); tt = sum(v[1] for v in per.values())
    row = {k: v[1] / max(1, v[0]) for k, v in per.items()}
    print(f"{label or tokenizer_path}: overall {tt/max(1,tw):.3f} tokens/word  "
          + "  ".join(f"{k.replace('train_','')}={v:.3f}" for k, v in row.items()))
    return tt / max(1, tw), row


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_dir", required=True)
    ap.add_argument("--out", default="artifacts/tokenizer.json")
    ap.add_argument("--vocab_size", type=int, default=32000)
    ap.add_argument("--min_frequency", type=int, default=2)
    ap.add_argument("--cache_dir", default="artifacts/tok_sample")
    ap.add_argument("--target_mb_per_source", type=float, default=60)
    ap.add_argument("--stop_after_docs", type=int, default=0)
    args = ap.parse_args()
    tr, ev = ensure_sample(args.train_dir, args.cache_dir, args.target_mb_per_source, args.stop_after_docs)
    print("Training BPE tokenizer...")
    train_tokenizer(tr, args.out, args.vocab_size, args.min_frequency)
    report(args.out, ev, label=f"vocab={args.vocab_size}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""
Train any of the 4 architectures on the pre-tokenized legal corpus (see prepare_data.py).

    python train.py --architecture decoder_only --data_dir artifacts/data --hidden_size 768 \
        --num_layers 12 --num_heads 12 --num_kv_heads 4 --context_length 1024 --max_steps 5000
"""
import argparse
import json
import math
import os
import time
from dataclasses import asdict

import torch
from torch.utils.data import DataLoader

from config import PathConfig, DataConfig, ModelConfig, TrainConfig
from data_utils import (TokenStore, sample_blocks, PackedLMDataset, WindowSeq2SeqDataset,
                        collate_lm, collate_seq2seq)
from models import build_model, count_params, get_amp_dtype


def get_lr(step, cfg: TrainConfig):
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / max(1, cfg.warmup_steps)
    progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    progress = min(progress, 1.0)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return cfg.min_lr + (cfg.lr - cfg.min_lr) * cosine


@torch.no_grad()
def evaluate(model, loader, device, architecture, amp_dtype=None, max_batches=None):
    model.eval()
    total_loss, n = 0.0, 0
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        with torch.autocast(device_type="cuda", dtype=amp_dtype or torch.float16,
                            enabled=(device == "cuda" and amp_dtype is not None)):
            if architecture == "enc_dec":
                _, loss = model(batch["src_ids"].to(device), batch["tgt_ids"].to(device),
                                labels=batch["labels"].to(device),
                                src_padding_mask=batch["src_padding_mask"].to(device))
            else:
                _, loss = model(batch["input_ids"].to(device), labels=batch["labels"].to(device))
        total_loss += loss.item()
        n += 1
    model.train()
    if n == 0:
        raise RuntimeError("validation loader produced no batches (val set smaller than one batch)")
    avg = total_loss / n
    return avg, math.exp(min(avg, 20))


def save_ckpt(path, model, mcfg, step, context_length):
    torch.save({"model": model.state_dict(), "config": asdict(mcfg), "step": step,
                "context_length": context_length}, path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--architecture", default="decoder_only",
                    choices=["decoder_only", "enc_dec", "efficient", "chunk_recurrent"])
    ap.add_argument("--data_dir", default=PathConfig.data_dir)
    ap.add_argument("--checkpoint_dir", default=PathConfig.checkpoint_dir)
    ap.add_argument("--tag", default=None, help="label used in artifacts/results.jsonl")
    ap.add_argument("--hidden_size", type=int, default=ModelConfig.hidden_size)
    ap.add_argument("--num_layers", type=int, default=ModelConfig.num_layers)
    ap.add_argument("--num_heads", type=int, default=ModelConfig.num_heads)
    ap.add_argument("--num_kv_heads", type=int, default=ModelConfig.num_kv_heads)
    ap.add_argument("--context_length", type=int, default=DataConfig.context_length)
    ap.add_argument("--local_window", type=int, default=ModelConfig.local_window)
    ap.add_argument("--global_every", type=int, default=ModelConfig.global_every)
    ap.add_argument("--chunk_size", type=int, default=ModelConfig.chunk_size)
    ap.add_argument("--max_train_tokens", type=int, default=DataConfig.max_train_tokens)
    ap.add_argument("--eval_tokens", type=int, default=400_000,
                    help="size of the fixed validation set (tokens); same for every run so val_loss is comparable")
    ap.add_argument("--batch_size", type=int, default=TrainConfig.batch_size)
    ap.add_argument("--grad_accum_steps", type=int, default=TrainConfig.grad_accum_steps)
    ap.add_argument("--lr", type=float, default=TrainConfig.lr)
    ap.add_argument("--max_steps", type=int, default=TrainConfig.max_steps)
    ap.add_argument("--warmup_steps", type=int, default=TrainConfig.warmup_steps)
    ap.add_argument("--eval_every", type=int, default=TrainConfig.eval_every)
    ap.add_argument("--save_every", type=int, default=0, help="0 = don't write intermediate step checkpoints")
    ap.add_argument("--log_every", type=int, default=TrainConfig.log_every)
    ap.add_argument("--dropout", type=float, default=ModelConfig.dropout)
    ap.add_argument("--weight_decay", type=float, default=TrainConfig.weight_decay)
    ap.add_argument("--source_weights", type=str, default=None, help='JSON, e.g. \'{"train_acts":0.5}\'')
    ap.add_argument("--early_stopping_patience", type=int, default=0)
    ap.add_argument("--early_stopping_min_delta", type=float, default=1e-3)
    ap.add_argument("--time_budget_min", type=float, default=0, help="stop training after this many minutes (0 = off)")
    ap.add_argument("--no_save", action="store_true", help="write no checkpoints (ablations that only need val_loss)")
    ap.add_argument("--save_final", type=int, default=1, help="also write <arch>_final.pt")
    ap.add_argument("--min_params", type=float, default=50e6)
    ap.add_argument("--max_params", type=float, default=300e6)
    ap.add_argument("--num_workers", type=int, default=2)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype = get_amp_dtype()
    if not args.no_save:
        os.makedirs(args.checkpoint_dir, exist_ok=True)
    tag = args.tag or args.architecture

    store = TokenStore(args.data_dir)
    source_weights = DataConfig().source_weights
    if args.source_weights:
        source_weights.update(json.loads(args.source_weights))

    is_s2s = args.architecture == "enc_dec"
    mcfg = ModelConfig(
        architecture=args.architecture, vocab_size=store.vocab_size,
        hidden_size=args.hidden_size, num_layers=args.num_layers,
        num_heads=args.num_heads, num_kv_heads=args.num_kv_heads,
        # RoPE models: generation crops to the length they were trained on.
        # enc_dec uses learned positions: must cover SRC_LEN(600) and TGT_LEN(224).
        max_position_embeddings=640 if is_s2s else args.context_length,
        local_window=args.local_window, global_every=args.global_every,
        chunk_size=args.chunk_size, dropout=args.dropout,
    )
    model = build_model(mcfg).to(device)
    n_params = count_params(model)
    print(f"[{tag}] {args.architecture} parameters: {n_params/1e6:.1f}M | device={device} amp={amp_dtype}", flush=True)
    assert args.min_params <= n_params <= args.max_params, (
        f"Model has {n_params/1e6:.1f}M params -- outside {args.min_params/1e6:.0f}M-{args.max_params/1e6:.0f}M; "
        f"adjust hidden_size/num_layers.")

    t_data = time.time()
    if is_s2s:
        train_ds = WindowSeq2SeqDataset(store, "train", source_weights, max(2000, args.max_train_tokens // 512), DataConfig.seed)
        val_ds = WindowSeq2SeqDataset(store, "val", None, max(64, args.eval_tokens // 512), DataConfig.seed + 1)
        collate = lambda b: collate_seq2seq(b, pad_id=store.pad_id)
    else:
        train_ds = PackedLMDataset(sample_blocks(store, "train", source_weights, args.context_length,
                                                 args.max_train_tokens, DataConfig.seed))
        val_ds = PackedLMDataset(sample_blocks(store, "val", None, args.context_length,
                                               args.eval_tokens, DataConfig.seed + 1))
        collate = collate_lm
    print(f"data ready in {time.time()-t_data:.0f}s | train items {len(train_ds):,} | val items {len(val_ds):,}", flush=True)
    if len(val_ds) < args.batch_size:
        raise RuntimeError(f"val set ({len(val_ds)}) smaller than batch_size ({args.batch_size}); raise --eval_tokens")

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate,
                              drop_last=True, num_workers=args.num_workers, persistent_workers=args.num_workers > 0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate, drop_last=True)

    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
    optim = torch.optim.AdamW([{"params": decay, "weight_decay": args.weight_decay},
                               {"params": no_decay, "weight_decay": 0.0}],
                              lr=args.lr, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and amp_dtype == torch.float16))
    tcfg = TrainConfig(lr=args.lr, min_lr=args.lr * 0.1, max_steps=args.max_steps, warmup_steps=args.warmup_steps)

    step, t0 = 0, time.time()
    model.train()
    train_iter = iter(train_loader)
    best_val, evals_since_improve = float("inf"), 0
    last_val = (float("nan"), float("nan"))
    stop_reason = None
    tok_per_step = args.batch_size * args.grad_accum_steps * (args.context_length if not is_s2s else 512)

    ref_val = float("inf")   # reference for the early-stopping patience counter (needs min_delta of progress)

    def do_eval():
        nonlocal best_val, ref_val, evals_since_improve, last_val, stop_reason
        val_loss, val_ppl = evaluate(model, val_loader, device, args.architecture, amp_dtype)
        last_val = (val_loss, val_ppl)
        print(f"  [eval] step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}", flush=True)
        if val_loss < best_val:                       # best checkpoint = truly lowest val_loss
            best_val = val_loss
            if not args.no_save:
                save_ckpt(os.path.join(args.checkpoint_dir, f"{args.architecture}_best.pt"),
                          model, mcfg, step, args.context_length)
        if val_loss < ref_val - args.early_stopping_min_delta:
            ref_val, evals_since_improve = val_loss, 0
        else:
            evals_since_improve += 1
            if args.early_stopping_patience > 0 and evals_since_improve >= args.early_stopping_patience:
                stop_reason = f"early stop (no val improvement for {evals_since_improve} evals)"

    last_eval_step = -1
    while step < args.max_steps and stop_reason is None:
        optim.zero_grad(set_to_none=True)
        for _ in range(args.grad_accum_steps):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)
            with torch.autocast(device_type="cuda", dtype=amp_dtype or torch.float16,
                                enabled=(device == "cuda" and amp_dtype is not None)):
                if is_s2s:
                    _, loss = model(batch["src_ids"].to(device), batch["tgt_ids"].to(device),
                                    labels=batch["labels"].to(device),
                                    src_padding_mask=batch["src_padding_mask"].to(device))
                else:
                    _, loss = model(batch["input_ids"].to(device), labels=batch["labels"].to(device))
                loss = loss / args.grad_accum_steps
            scaler.scale(loss).backward()

        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), TrainConfig.grad_clip)
        lr = get_lr(step, tcfg)
        for g in optim.param_groups:
            g["lr"] = lr
        scaler.step(optim)
        scaler.update()
        step += 1

        if step % args.log_every == 0 or step == 1:
            l = loss.item() * args.grad_accum_steps
            if not math.isfinite(l):
                raise RuntimeError(f"loss became {l} at step {step} (lr={lr:.2e}); lower --lr or check the data")
            el = time.time() - t0
            print(f"step {step}/{args.max_steps} | loss {l:.4f} | lr {lr:.2e} | {el:.0f}s | "
                  f"{step*tok_per_step/el/1e3:.1f}k tok/s", flush=True)

        if step % args.eval_every == 0:
            do_eval(); last_eval_step = step
        if args.save_every and step % args.save_every == 0 and not args.no_save:
            save_ckpt(os.path.join(args.checkpoint_dir, f"{args.architecture}_step{step}.pt"),
                      model, mcfg, step, args.context_length)
        if args.time_budget_min and (time.time() - t0) > args.time_budget_min * 60 and stop_reason is None:
            stop_reason = f"time budget of {args.time_budget_min:.0f} min reached"

    if last_eval_step != step:
        do_eval()
    if not args.no_save and args.save_final:
        save_ckpt(os.path.join(args.checkpoint_dir, f"{args.architecture}_final.pt"),
                  model, mcfg, step, args.context_length)

    secs = time.time() - t0
    print(f"Training complete{' (' + stop_reason + ')' if stop_reason else ''}. "
          f"best_val_loss={best_val:.4f} final_val_loss={last_val[0]:.4f}", flush=True)
    os.makedirs("artifacts", exist_ok=True)
    with open("artifacts/results.jsonl", "a") as f:
        f.write(json.dumps({
            "kind": "train", "tag": tag, "architecture": args.architecture,
            "params_M": round(n_params / 1e6, 1), "context_length": args.context_length,
            "steps": step, "best_val_loss": best_val, "best_val_ppl": math.exp(min(best_val, 20)),
            "final_val_loss": last_val[0], "minutes": round(secs / 60, 1),
            "tokens_seen_M": round(step * tok_per_step / 1e6, 1),
            "dropout": args.dropout, "weight_decay": args.weight_decay,
            "source_weights": source_weights, "stop": stop_reason,
        }) + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile generate.py
"""
Generate 128-word continuations for every test prompt and write prediction.json (+ submission.zip).

Fixes vs the old version:
  * the prompt is encoded WITHOUT a trailing <eos> (the old tokenizer template appended one, so the
    model was told the document had ended and continued with an unrelated document);
  * decoder-style models get [<bos>] + prompt, matching how training documents start;
  * enc_dec receives the LAST 600 prompt tokens (nearest to the continuation), not the first 600;
  * torch.load works on torch>=2.6; fp16/bf16 autocast for speed; partial results are written so a
    session timeout does not lose everything (--resume).

Usage:
    python generate.py --checkpoint artifacts/ckpt_tier2/decoder_only_best.pt \
        --tokenizer_path artifacts/tokenizer.json --test_file test.json
"""
import argparse
import json
import os
import time
import zipfile

import torch
from tokenizers import Tokenizer

from config import GenConfig
from models import load_checkpoint, get_amp_dtype
from data_utils import SRC_LEN


def load_test_prompts(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)  # {"1": {"text": ...}, "2": {"text": ...}, ...}


def trim_to_words(text, n_words):
    return " ".join(text.split()[:n_words])


def _new_tokens(gcfg):
    return int(gcfg.max_new_words * 1.7)     # ~1.25 tokens/word, overshoot then trim to words


def generate_decoder_only(model, tokenizer, prompt, device, gcfg):
    bos_id = tokenizer.token_to_id("<bos>")
    ids = [bos_id] + tokenizer.encode(prompt, add_special_tokens=False).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(
        input_ids, max_new_tokens=_new_tokens(gcfg),
        temperature=gcfg.temperature, top_p=gcfg.top_p, top_k=gcfg.top_k,
        repetition_penalty=gcfg.repetition_penalty,
        no_repeat_ngram_size=gcfg.no_repeat_ngram_size,
        eos_token_id=None,      # never stop early: we always want ~128 words
    )
    new_tokens = out[0, input_ids.size(1):].tolist()
    return trim_to_words(tokenizer.decode(new_tokens, skip_special_tokens=True), gcfg.max_new_words)


def generate_enc_dec(model, tokenizer, prompt, device, gcfg):
    src_ids = tokenizer.encode(prompt, add_special_tokens=False).ids[-SRC_LEN:]
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    out = model.generate(
        src, bos_id=bos_id, eos_id=eos_id, max_new_tokens=min(_new_tokens(gcfg), 220),
        temperature=gcfg.temperature, top_p=gcfg.top_p, top_k=gcfg.top_k,
        repetition_penalty=gcfg.repetition_penalty,
        no_repeat_ngram_size=gcfg.no_repeat_ngram_size,
    )
    new_tokens = out[0, 1:].tolist()  # drop leading <bos>
    return trim_to_words(tokenizer.decode(new_tokens, skip_special_tokens=True), gcfg.max_new_words)


def get_gen_fn(architecture):
    return generate_enc_dec if architecture == "enc_dec" else generate_decoder_only


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--tokenizer_path", default="artifacts/tokenizer.json")
    ap.add_argument("--test_file", required=True)
    ap.add_argument("--out", default="artifacts/prediction.json")
    ap.add_argument("--zip_out", default="artifacts/submission.zip")
    ap.add_argument("--temperature", type=float, default=GenConfig.temperature)
    ap.add_argument("--top_p", type=float, default=GenConfig.top_p)
    ap.add_argument("--top_k", type=int, default=GenConfig.top_k)
    ap.add_argument("--repetition_penalty", type=float, default=GenConfig.repetition_penalty)
    ap.add_argument("--limit", type=int, default=0, help="only the first N prompts (smoke test)")
    ap.add_argument("--resume", action="store_true", help="continue from a partially written --out")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = get_amp_dtype()
    tokenizer = Tokenizer.from_file(args.tokenizer_path)
    model, mcfg, ckpt = load_checkpoint(args.checkpoint, device)
    print(f"Loaded {mcfg.architecture} checkpoint from step {ckpt.get('step')}", flush=True)

    gcfg = GenConfig(temperature=args.temperature, top_p=args.top_p,
                     top_k=args.top_k, repetition_penalty=args.repetition_penalty)
    gen_fn = get_gen_fn(mcfg.architecture)

    prompts = load_test_prompts(args.test_file)
    items = list(prompts.items())[: args.limit or None]
    predictions = {}
    if args.resume and os.path.exists(args.out):
        with open(args.out, encoding="utf-8") as f:
            predictions = json.load(f)
        print(f"resuming: {len(predictions)} predictions already done", flush=True)
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)

    def dump():
        with open(args.out + ".tmp", "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        os.replace(args.out + ".tmp", args.out)

    t0 = time.time()
    todo = [(k, r) for k, r in items if k not in predictions]
    for i, (key, rec) in enumerate(todo):
        with torch.no_grad(), torch.autocast("cuda", dtype=amp or torch.float16, enabled=(device == "cuda" and amp is not None)):
            predictions[key] = {"text": gen_fn(model, tokenizer, rec["text"], device, gcfg)}
        if (i + 1) % 10 == 0:
            el = time.time() - t0
            print(f"generated {i + 1}/{len(todo)} | {el/(i+1):.1f}s each | ETA {el/(i+1)*(len(todo)-i-1)/60:.0f} min", flush=True)
        if (i + 1) % 25 == 0:
            dump()
    dump()
    print(f"Wrote {args.out} ({len(predictions)} predictions)")

    with zipfile.ZipFile(args.zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(args.out, arcname="prediction.json")     # the name the sanity check / spec 7.2.5 expects
    print(f"Wrote {args.zip_out} (ready for Codabench)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eval.py
"""
Score a checkpoint's generations on held-out (prompt, reference) pairs with the competition metrics
(BERTScore F1 primary; ROUGE-L F1 and BLEU-4 secondary, spec 7.8). Pairs come from
<data_dir>/val_pairs.json, which prepare_data.py built from validation documents only, so this never
touches the official test set and never overlaps the training data.

    python eval.py --checkpoint artifacts/ckpt_tier2/decoder_only_best.pt --n_examples 60 --tag t2_best
"""
import argparse
import json
import os
import random
import time

import torch
from tokenizers import Tokenizer

from config import GenConfig, PathConfig
from generate import get_gen_fn
from models import load_checkpoint, get_amp_dtype


def compute_metrics(preds, refs, device, skip_bertscore=False):
    results = {"bertscore_f1": None, "rougeL_f1": None, "bleu4": None}
    preds = [p if p.strip() else "." for p in preds]   # metric libs choke on empty strings

    if not skip_bertscore:
        try:
            from bert_score import score as bert_score
            _, _, F1 = bert_score(preds, refs, lang="en", verbose=False, device=device, batch_size=32)
            results["bertscore_f1"] = F1.mean().item()
        except Exception as e:      # missing package, no internet for the roberta download, ...
            print(f"[warn] BERTScore skipped: {type(e).__name__}: {e}")
    try:
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
        s = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
        results["rougeL_f1"] = sum(s) / len(s)
    except Exception as e:
        print(f"[warn] ROUGE-L skipped: {type(e).__name__}: {e}")
    try:
        import sacrebleu
        results["bleu4"] = sacrebleu.corpus_bleu(preds, [refs]).score   # sacrebleu default = BLEU-4
    except Exception as e:
        print(f"[warn] BLEU skipped: {type(e).__name__}: {e}")
    return results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--tokenizer_path", default=PathConfig.tokenizer_out)
    ap.add_argument("--data_dir", default=PathConfig.data_dir)
    ap.add_argument("--n_examples", type=int, default=60)
    ap.add_argument("--temperature", type=float, default=GenConfig.temperature)
    ap.add_argument("--top_p", type=float, default=GenConfig.top_p)
    ap.add_argument("--top_k", type=int, default=GenConfig.top_k)
    ap.add_argument("--repetition_penalty", type=float, default=GenConfig.repetition_penalty)
    ap.add_argument("--seed", type=int, default=123)
    ap.add_argument("--tag", default=None)
    ap.add_argument("--skip_bertscore", action="store_true")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = get_amp_dtype()
    torch.manual_seed(args.seed)
    tokenizer = Tokenizer.from_file(args.tokenizer_path)
    model, mcfg, ckpt = load_checkpoint(args.checkpoint, device)
    arch = mcfg.architecture
    print(f"Loaded {arch} checkpoint from step {ckpt.get('step')}", flush=True)

    with open(os.path.join(args.data_dir, "val_pairs.json"), encoding="utf-8") as f:
        pairs = json.load(f)
    rng = random.Random(args.seed)
    rng.shuffle(pairs)
    pairs = pairs[: args.n_examples]          # same seed -> same pairs for every eval call
    assert pairs, "val_pairs.json is empty"

    gcfg = GenConfig(temperature=args.temperature, top_p=args.top_p,
                     top_k=args.top_k, repetition_penalty=args.repetition_penalty)
    gen_fn = get_gen_fn(arch)

    preds, refs = [], []
    t0 = time.time()
    for i, p in enumerate(pairs):
        with torch.no_grad(), torch.autocast("cuda", dtype=amp or torch.float16, enabled=(device == "cuda" and amp is not None)):
            preds.append(gen_fn(model, tokenizer, p["prompt"], device, gcfg))
        refs.append(p["reference"])
        if (i + 1) % 20 == 0:
            print(f"  generated {i+1}/{len(pairs)} ({(time.time()-t0)/(i+1):.1f}s each)", flush=True)

    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    metrics = compute_metrics(preds, refs, device, args.skip_bertscore)

    print("\n=== Results ===")
    print(f"architecture: {arch}\ncheckpoint:   {args.checkpoint}")
    print(f"decoding:     temp={args.temperature} top_p={args.top_p} top_k={args.top_k} rep_penalty={args.repetition_penalty}")
    print(f"n_examples:   {len(preds)}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}" if v is not None else f"{k}: (skipped)")
    print("\n--- sample ---")
    print("PROMPT (tail):", pairs[0]["prompt"][-200:])
    print("REFERENCE:", refs[0])
    print("PREDICTION:", preds[0])

    os.makedirs("artifacts", exist_ok=True)
    with open("artifacts/results.jsonl", "a") as f:
        f.write(json.dumps({
            "kind": "eval", "tag": args.tag or os.path.basename(args.checkpoint), "architecture": arch,
            "checkpoint": args.checkpoint, "n_examples": len(preds),
            "temperature": args.temperature, "top_p": args.top_p, "top_k": args.top_k,
            "repetition_penalty": args.repetition_penalty, **metrics,
        }) + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile distill.py
"""
Model compression via knowledge distillation (spec 7.4). Trains a smaller student to match a trained
teacher's output distribution (KL on soft targets) plus hard-label cross-entropy on the corpus.

Supports the packed-LM architectures (decoder_only / efficient / chunk_recurrent). The student
is an experimental compact artifact for the size/quality trade-off study (spec 7.4/7.9); it does not
need to sit in the 50M-300M submission range.

    python distill.py --teacher_checkpoint artifacts/ckpt_tier2/decoder_only_best.pt \
        --student_hidden_size 384 --student_num_layers 6 --max_steps 3000
"""
import argparse
import json
import math
import os
import time
from dataclasses import asdict

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from config import PathConfig, DataConfig, ModelConfig, TrainConfig
from data_utils import TokenStore, sample_blocks, PackedLMDataset, collate_lm
from models import build_model, count_params, load_checkpoint, get_amp_dtype
from train import get_lr, evaluate


def distill_loss(student_logits, teacher_logits, labels, temperature, alpha):
    """alpha weights the soft-target KD loss; (1-alpha) weights hard-label CE. Computed in fp32."""
    V = student_logits.size(-1)
    s_all = student_logits[:, :-1].float().reshape(-1, V)
    t_all = teacher_logits[:, :-1].float().reshape(-1, V)
    ce = F.cross_entropy(s_all, labels[:, 1:].reshape(-1), ignore_index=-100)
    kd = F.kl_div(F.log_softmax(s_all / temperature, dim=-1), F.softmax(t_all / temperature, dim=-1),
                  reduction="batchmean") * (temperature ** 2)
    return alpha * kd + (1 - alpha) * ce, ce.item(), kd.item()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--teacher_checkpoint", required=True)
    ap.add_argument("--data_dir", default=PathConfig.data_dir)
    ap.add_argument("--checkpoint_dir", default="artifacts/ckpt_distill")
    ap.add_argument("--student_hidden_size", type=int, default=384)
    ap.add_argument("--student_num_layers", type=int, default=6)
    ap.add_argument("--student_num_heads", type=int, default=6)
    ap.add_argument("--student_num_kv_heads", type=int, default=3)
    ap.add_argument("--context_length", type=int, default=0, help="0 = same as the teacher was trained with")
    ap.add_argument("--max_train_tokens", type=int, default=20_000_000)
    ap.add_argument("--eval_tokens", type=int, default=400_000)
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--grad_accum_steps", type=int, default=4)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--max_steps", type=int, default=3000)
    ap.add_argument("--warmup_steps", type=int, default=200)
    ap.add_argument("--temperature", type=float, default=2.0, help="KD softmax temperature")
    ap.add_argument("--alpha", type=float, default=0.7, help="weight on KD loss vs hard-label CE")
    ap.add_argument("--eval_every", type=int, default=300)
    ap.add_argument("--log_every", type=int, default=50)
    ap.add_argument("--tag", default="distilled_student")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = get_amp_dtype()
    use_amp = device == "cuda" and amp is not None
    os.makedirs(args.checkpoint_dir, exist_ok=True)

    print("Loading teacher...", flush=True)
    teacher, tcfg_model, tckpt = load_checkpoint(args.teacher_checkpoint, device)
    if tcfg_model.architecture == "enc_dec":
        raise SystemExit("distill.py supports decoder_only / efficient / chunk_recurrent teachers only")
    for p in teacher.parameters():
        p.requires_grad = False
    ctx = args.context_length or tckpt.get("context_length") or tcfg_model.max_position_embeddings
    n_teacher = count_params(teacher)
    print(f"Teacher ({tcfg_model.architecture}): {n_teacher/1e6:.1f}M params | context {ctx}", flush=True)

    scfg = ModelConfig(
        architecture=tcfg_model.architecture, vocab_size=tcfg_model.vocab_size,
        hidden_size=args.student_hidden_size, num_layers=args.student_num_layers,
        num_heads=args.student_num_heads, num_kv_heads=args.student_num_kv_heads,
        max_position_embeddings=tcfg_model.max_position_embeddings,
        local_window=tcfg_model.local_window, global_every=tcfg_model.global_every,
        chunk_size=tcfg_model.chunk_size,
    )
    student = build_model(scfg).to(device)
    n_student = count_params(student)
    print(f"Student: {n_student/1e6:.1f}M params ({n_student/n_teacher*100:.1f}% of teacher size)", flush=True)

    store = TokenStore(args.data_dir)
    train_ds = PackedLMDataset(sample_blocks(store, "train", DataConfig().source_weights, ctx,
                                             args.max_train_tokens, DataConfig.seed))
    val_ds = PackedLMDataset(sample_blocks(store, "val", None, ctx, args.eval_tokens, DataConfig.seed + 1))
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_lm,
                              drop_last=True, num_workers=2, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_lm, drop_last=True)

    decay = [p for p in student.parameters() if p.dim() >= 2]
    no_decay = [p for p in student.parameters() if p.dim() < 2]
    optim = torch.optim.AdamW([{"params": decay, "weight_decay": TrainConfig.weight_decay},
                               {"params": no_decay, "weight_decay": 0.0}], lr=args.lr, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and amp == torch.float16))
    tcfg = TrainConfig(lr=args.lr, min_lr=args.lr * 0.1, max_steps=args.max_steps, warmup_steps=args.warmup_steps)

    step, t0 = 0, time.time()
    student.train()
    teacher.eval()
    train_iter = iter(train_loader)
    best_val = float("inf")

    while step < args.max_steps:
        optim.zero_grad(set_to_none=True)
        for _ in range(args.grad_accum_steps):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            with torch.autocast("cuda", dtype=amp or torch.float16, enabled=use_amp):
                with torch.no_grad():
                    teacher_logits, _ = teacher(input_ids)
                student_logits, _ = student(input_ids)
            loss, ce, kd = distill_loss(student_logits, teacher_logits, labels, args.temperature, args.alpha)
            scaler.scale(loss / args.grad_accum_steps).backward()

        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(student.parameters(), TrainConfig.grad_clip)
        lr = get_lr(step, tcfg)
        for g in optim.param_groups:
            g["lr"] = lr
        scaler.step(optim)
        scaler.update()
        step += 1

        if step % args.log_every == 0:
            if not math.isfinite(ce):
                raise RuntimeError(f"non-finite loss at step {step}")
            print(f"step {step}/{args.max_steps} | ce {ce:.4f} | kd {kd:.4f} | lr {lr:.2e} | {time.time()-t0:.0f}s", flush=True)
        if step % args.eval_every == 0 or step == args.max_steps:
            v, ppl = evaluate(student, val_loader, device, tcfg_model.architecture, amp)
            best_val = min(best_val, v)
            print(f"  [eval] step {step} | student val_loss {v:.4f} | val_ppl {ppl:.2f}", flush=True)

    # teacher val loss on the same val set, for the size/quality comparison
    t_val, t_ppl = evaluate(teacher, val_loader, device, tcfg_model.architecture, amp)
    s_val, s_ppl = evaluate(student, val_loader, device, tcfg_model.architecture, amp)
    out_path = os.path.join(args.checkpoint_dir, f"{tcfg_model.architecture}_student_distilled.pt")
    torch.save({"model": student.state_dict(), "config": asdict(scfg), "step": step, "context_length": ctx}, out_path)
    print(f"Saved distilled student to {out_path} ({n_student/1e6:.1f}M vs teacher {n_teacher/1e6:.1f}M)")
    print(f"val_loss  teacher {t_val:.4f} (ppl {t_ppl:.2f})  |  student {s_val:.4f} (ppl {s_ppl:.2f})")
    with open("artifacts/results.jsonl", "a") as f:
        f.write(json.dumps({"kind": "train", "tag": args.tag, "architecture": tcfg_model.architecture,
                            "params_M": round(n_student / 1e6, 1), "context_length": ctx, "steps": step,
                            "best_val_loss": s_val, "best_val_ppl": s_ppl, "final_val_loss": s_val,
                            "teacher_params_M": round(n_teacher / 1e6, 1), "teacher_val_loss": t_val}) + "\n")


if __name__ == "__main__":
    main()

## 3. Helpers — `run()` fails loudly, results table

In [ ]:
import json, shlex, subprocess, sys, time
import pandas as pd
from IPython.display import display

def run(cmd):
    """Run a command, stream its output, and STOP THE NOTEBOOK if it fails.
    (`!python ...` ignores the exit code, which is how the last run limped on after every training process
    was OOM-killed.)"""
    args = shlex.split(cmd)
    if args[0] == "python":
        args[0] = sys.executable
    print("$", " ".join(args)[:400], flush=True)
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         cwd="/kaggle/working")
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit {rc} after {(time.time()-t0)/60:.1f} min]", flush=True)
    if rc != 0:
        hint = "  <- SIGKILL: the process was almost certainly killed for running out of memory" if rc == -9 else ""
        raise RuntimeError(f"command failed with exit code {rc}{hint}\n{cmd[:300]}")

def load_results():
    p = "artifacts/results.jsonl"
    rows = [json.loads(l) for l in open(p)] if os.path.exists(p) else []
    df = pd.DataFrame(rows)
    return df[~df.tag.str.startswith("smoke")] if len(df) else df

def show(kind, prefix=None, cols=None):
    df = load_results()
    if not len(df):
        print("(no results yet)"); return
    df = df[df.kind == kind]
    if prefix:
        df = df[df.tag.str.startswith(prefix)]
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    display(df.dropna(axis=1, how="all").reset_index(drop=True))

TRAIN_COLS = ["tag", "architecture", "params_M", "context_length", "steps", "best_val_loss", "best_val_ppl", "minutes", "tokens_seen_M", "stop"]
EVAL_COLS = ["tag", "architecture", "n_examples", "bertscore_f1", "rougeL_f1", "bleu4", "temperature", "top_p", "top_k", "repetition_penalty"]

# Architecture shapes. Tier-1: all four inside the 50M-300M limit and roughly comparable (51-62M).
ARCH_T1 = {
    "decoder_only":    "--hidden_size 640 --num_layers 8  --num_heads 10 --num_kv_heads 5",                                   # 57.4M
    "enc_dec":         "--hidden_size 512 --num_layers 12 --num_heads 8  --num_kv_heads 4",                                   # 61.6M (6 enc + 6 dec)
    "efficient":       "--hidden_size 640 --num_layers 8  --num_heads 10 --num_kv_heads 5 --local_window 128 --global_every 4",  # 57.4M
    "chunk_recurrent": "--hidden_size 512 --num_layers 8  --num_heads 8  --num_kv_heads 4 --chunk_size 128",                  # 51.3M
}
# Tier-2 (full run) shapes, all < 300M.
ARCH_T2 = {
    "decoder_only":    "--hidden_size 768 --num_layers 12 --num_heads 12 --num_kv_heads 4",                                   # 101.9M
    "efficient":       "--hidden_size 768 --num_layers 12 --num_heads 12 --num_kv_heads 4 --local_window 128 --global_every 4",  # 101.9M
    "chunk_recurrent": "--hidden_size 640 --num_layers 10 --num_heads 10 --num_kv_heads 5 --chunk_size 128",                  # 87.5M
    "enc_dec":         "--hidden_size 640 --num_layers 12 --num_heads 10 --num_kv_heads 5",                                   # 91.0M
}
print("helpers ready")

## 4. Tokenizer design experiment (§7.3, §7.4)

§7.3 allows reusing Assignment 1's BPE, but a pure-Python merge loop is not viable for a multi-billion-token corpus, so we use Hugging Face's Rust BPE trainer (same algorithm; the spec permits it).

The first call streams all four files once and caches a random ~60 MB/source sample (`artifacts/tok_sample/`); the other vocab sizes reuse that cache. We compare **tokens/word on held-out sample documents** (docs the trainer never saw), overall and per source.

In [ ]:
import shutil
if FORCE_REBUILD:
    for p in ["artifacts/tok_sample", "artifacts/tokenizer.json", "artifacts/tok_16k.json", "artifacts/tok_50k.json", DATA_DIR]:
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))

TOK_ARGS = f"--train_dir {TRAIN_DIR} --target_mb_per_source {S['tok_mb']} --stop_after_docs {S['tok_docs']}"
# 32k = the tokenizer we will actually use (also builds the cached sample on first call)
if not os.path.exists("artifacts/tokenizer.json"):
    run(f"python tokenizer_train.py {TOK_ARGS} --vocab_size 32000 --out artifacts/tokenizer.json")
else:
    print("artifacts/tokenizer.json exists -- skipping (set FORCE_REBUILD=True to redo)")
if not QUICK:   # the vocab-size comparison (spec 7.4); skipped in QUICK mode
    for v, f in [(16000, "tok_16k"), (50000, "tok_50k")]:
        if not os.path.exists(f"artifacts/{f}.json"):
            run(f"python tokenizer_train.py {TOK_ARGS} --vocab_size {v} --out artifacts/{f}.json")

In [ ]:
from tokenizer_train import report
cands = [("artifacts/tokenizer.json", "vocab=32000")] + \
        [(f"artifacts/{f}.json", f"vocab={v}") for v, f in [(16000, "tok_16k"), (50000, "tok_50k")] if os.path.exists(f"artifacts/{f}.json")]
for path, label in cands:
    report(path, "artifacts/tok_sample/tok_eval.jsonl", label=label)
from tokenizers import Tokenizer
tk = Tokenizer.from_file("artifacts/tokenizer.json")
print("special ids:", {t: tk.token_to_id(t) for t in ["<pad>", "<bos>", "<eos>", "<unk>"]}, "| post-processor:", tk.post_processor)
sample = "The learned counsel for the appellant submitted that Section 34 of the Arbitration and Conciliation Act, 1996 (\u20b9 5,00,000)."
print("round trip ok:", tk.decode(tk.encode(sample).ids) == sample, "|", len(tk.encode(sample).ids), "tokens")

**Record the tokens/word numbers above in your report** — lower is more efficient, but a larger vocab also enlarges the embedding/output matrix and eats into the 50M–300M budget (50k × 768 = 38M parameters in the embedding alone). 32k is the compromise used below; change `--vocab_size` if your numbers say otherwise (then the parameter counts printed by `train.py` change too).

## 5. Tokenize the corpus once (streaming, constant memory)

Reads each 1–10 GB JSON file record by record, keeps a random subset of documents of the big sources (up to ~250M tokens per source; `train_acts` is kept whole), splits **by document** into train / val (val docs never appear in training), and writes `uint16` shards + a held-out set of (512-word prompt → next-128-word reference) pairs for generation-quality evaluation. Expect ~30–50 min for the full corpus (dominated by JSON parsing + tokenizing 19 GB).

In [ ]:
if os.path.exists(f"{DATA_DIR}/meta.json") and not FORCE_REBUILD:
    print("tokenized shards already exist -- skipping (set FORCE_REBUILD=True after switching QUICK<->FULL)")
else:
    run(f"python prepare_data.py --train_dir {TRAIN_DIR} --tokenizer_path artifacts/tokenizer.json --out_dir {DATA_DIR} "
        f"--max_tokens_per_source {PREP_CAP} --val_fraction {S['val_frac']} --stop_after_docs {S['prep_docs']}")

In [ ]:
# Sanity-check the shards BEFORE spending GPU time on them
import numpy as np
meta = json.load(open(f"{DATA_DIR}/meta.json"))
rows = [{"source": k, **{c: v[c] for c in ["docs_scanned", "train_docs", "val_docs", "train_tokens", "val_tokens", "val_pairs"]}}
        for k, v in meta["sources"].items()]
df = pd.DataFrame(rows); df["train_tokens_M"] = (df.train_tokens / 1e6).round(1); df["val_tokens_M"] = (df.val_tokens / 1e6).round(2)
display(df.drop(columns=["train_tokens", "val_tokens"]))
assert len(meta["sources"]) == 4 and (df.train_tokens > 0).all(), "a source produced no tokens"
if (df.val_tokens < 5_000).any():
    print("WARNING: some source has almost no validation tokens:", df[df.val_tokens < 5_000].source.tolist())
pairs = json.load(open(f"{DATA_DIR}/val_pairs.json"))
print("held-out generation pairs:", len(pairs), "| by source:", pd.Series([p["source"] for p in pairs]).value_counts().to_dict())
assert len(pairs) >= 4, "too few held-out (prompt, reference) pairs for eval.py"
# decode the first tokens of a training document
tok_arr = np.memmap(f"{DATA_DIR}/train_supreme_court.train.bin", dtype=np.uint16, mode="r")
print("first doc starts with <bos>:", tok_arr[0] == meta["bos_id"])
print(repr(tk.decode([int(x) for x in tok_arr[1:80]])))
print(subprocess.run(["du", "-sh", DATA_DIR], capture_output=True, text=True).stdout.strip())

## 6. Smoke test (~5 min): every architecture trains, evaluates and generates for a few steps

If anything is broken (shape error, OOM, bad import, wrong path) it fails **here**, in minutes, instead of an hour into a run.

In [ ]:
SMOKE = (f"--data_dir {DATA_DIR} --context_length 512 --max_train_tokens 1000000 --max_steps 12 --warmup_steps 3 "
         f"--eval_every 6 --log_every 6 --eval_tokens 60000 --batch_size 4 --grad_accum_steps 1 --num_workers 0 "
         f"--save_final 0 --checkpoint_dir artifacts/ckpt_smoke")
for arch, shape in ARCH_T1.items():
    run(f"python train.py --architecture {arch} {shape} {SMOKE} --tag smoke_{arch}")
for arch in ARCH_T1:
    run(f"python eval.py --checkpoint artifacts/ckpt_smoke/{arch}_best.pt --tokenizer_path artifacts/tokenizer.json "
        f"--data_dir {DATA_DIR} --n_examples 2 --skip_bertscore --tag smoke_eval_{arch}")
# official-format generation on 2 real test prompts
test = json.load(open(TEST_FILE))
print("test prompts:", len(test), "| first key:", next(iter(test)), "| words in first prompt:", len(next(iter(test.values()))["text"].split()))
json.dump(dict(list(test.items())[:2]), open("artifacts/smoke_test.json", "w"))
run("python generate.py --checkpoint artifacts/ckpt_smoke/decoder_only_best.pt --tokenizer_path artifacts/tokenizer.json "
    "--test_file artifacts/smoke_test.json --out artifacts/smoke_pred.json --zip_out artifacts/smoke.zip --limit 2")
shutil.rmtree("artifacts/ckpt_smoke", ignore_errors=True)
print("\nSMOKE TEST PASSED")

## 7. Tier-1: cheap architecture search (§7.3, §7.4)

All four architectures, same token budget, sizes inside the 50–300M limit. **Caveat for the report:** `val_loss` is only comparable among the three packed-LM models (they predict the same tokens); `enc_dec` predicts a 222-token target given a 600-token source, so compare it to the others with the generation metrics (BERTScore / ROUGE-L / BLEU-4) below, not with val_loss. (§7.9 asks you to study exactly this perplexity-vs-quality relationship.)

In [ ]:
def train_t1(arch):
    run(f"python train.py --architecture {arch} --data_dir {DATA_DIR} {ARCH_T1[arch]} --context_length 512 "
        f"--max_train_tokens {S['t1_tokens']} --max_steps {S['t1_steps']} --warmup_steps {S['t1_warm']} "
        f"--batch_size 16 --grad_accum_steps 2 --eval_every {S['t1_eval']} --log_every 50 "
        f"--eval_tokens {S['eval_tokens']} --save_final 0 --checkpoint_dir artifacts/ckpt_tier1 --tag t1_{arch}")
train_t1("decoder_only")

In [ ]:
train_t1("enc_dec")

In [ ]:
train_t1("efficient")

In [ ]:
train_t1("chunk_recurrent")

In [ ]:
show("train", "t1_", TRAIN_COLS)

Now check whether validation perplexity actually tracks generation quality (§7.9) — held-out prompt → 128-word continuation, scored with BERTScore / ROUGE-L / BLEU-4.

In [ ]:
for arch in ARCH_T1:
    run(f"python eval.py --checkpoint artifacts/ckpt_tier1/{arch}_best.pt --tokenizer_path artifacts/tokenizer.json "
        f"--data_dir {DATA_DIR} --n_examples {S['n_eval_t1']} --tag t1eval_{arch}")
show("eval", "t1eval_", EVAL_COLS)

## 8. Data-mixing ablation (§7.4)

Two short `decoder_only` runs: equal source weights vs. judgment-heavy (Supreme Court + High Courts up-weighted). The validation set is **identical** for both (equal-weight, fixed seed), so the `val_loss` values are directly comparable.

In [ ]:
ABL = (f"python train.py --architecture decoder_only --data_dir {DATA_DIR} {ARCH_T1['decoder_only']} "
       f"--max_train_tokens {S['abl_tokens']} --max_steps {S['abl_steps']} --warmup_steps {S['abl_warm']} "
       f"--eval_every {S['abl_eval']} --log_every 100 --eval_tokens {S['eval_tokens']}")
w_uniform = json.dumps({"train_acts": 1.0, "train_high_courts": 1.0, "train_supreme_court": 1.0, "train_tribunals": 1.0})
w_judg    = json.dumps({"train_acts": 0.5, "train_high_courts": 2.0, "train_supreme_court": 2.0, "train_tribunals": 0.5})
run(f"{ABL} --context_length 512 --batch_size 16 --grad_accum_steps 2 --source_weights '{w_uniform}' --no_save --tag abl_mix_uniform")

In [ ]:
run(f"{ABL} --context_length 512 --batch_size 16 --grad_accum_steps 2 --source_weights '{w_judg}' --no_save --tag abl_mix_judgment")
show("train", "abl_mix", TRAIN_COLS)

## 9. Context-length ablation (§7.4)

Same architecture and token budget, `context_length` 512 vs 1024. Note that a longer context lowers per-token loss *by construction*, so `val_loss` alone flatters 1024; the checkpoints are therefore also scored on the real task (prompt → continuation), which is what matters — the test prompts are ≈ 620 tokens, so a 512-token model cannot even see a whole prompt.

In [ ]:
run(f"{ABL} --context_length 512  --batch_size 16 --grad_accum_steps 2 --save_final 0 --checkpoint_dir artifacts/ckpt_abl_ctx512  --tag abl_ctx512")

In [ ]:
run(f"{ABL} --context_length 1024 --batch_size 8  --grad_accum_steps 4 --save_final 0 --checkpoint_dir artifacts/ckpt_abl_ctx1024 --tag abl_ctx1024")
show("train", "abl_ctx", TRAIN_COLS)

In [ ]:
for c in (512, 1024):
    run(f"python eval.py --checkpoint artifacts/ckpt_abl_ctx{c}/decoder_only_best.pt --tokenizer_path artifacts/tokenizer.json "
        f"--data_dir {DATA_DIR} --n_examples {S['n_eval_t1']} --tag abl_ctx{c}_eval")
show("eval", "abl_ctx", EVAL_COLS)

## 9b. Regularization ablation (§7.4)

Higher dropout + weight decay vs. the defaults. On a small token budget don't expect a big difference (the model is under-trained, not over-fit); noting *why* it is small at this scale is part of what §7.4/§7.9 ask for.

In [ ]:
run(f"{ABL} --context_length 512 --batch_size 16 --grad_accum_steps 2 --dropout 0.1 --weight_decay 0.1 --no_save --tag abl_reg_default")

In [ ]:
run(f"{ABL} --context_length 512 --batch_size 16 --grad_accum_steps 2 --dropout 0.3 --weight_decay 0.3 --no_save --tag abl_reg_heavy")
show("train", "abl_reg", TRAIN_COLS)

## 10. Pick a winner, then Tier-2: the full run (§7.4, §7.5)

Read the Tier-1 tables above and set `WINNER` (compare by **BERTScore/ROUGE-L**, not val_loss alone — see the caveat in §7). The final model uses `context_length = 1024` because a test prompt (~620 tokens) + a 128-word continuation (~170 tokens) does not fit in 512; the context ablation above should support this choice.

Sizing: ~100M parameters, 5 000 steps × 32 768 tokens ≈ 165M tokens, roughly 2.5–3 h on a T4 (less on an A100/H100). Early stopping is on (`--early_stopping_patience`) and `--time_budget_min` is a hard wall-clock guard so a slow GPU can't run you past Kaggle's 12 h limit. `train.py` enforces 50M ≤ params ≤ 300M.

In [ ]:
WINNER = "decoder_only"      # <- set to one of: decoder_only | enc_dec | efficient | chunk_recurrent
T2_CTX = 1024
assert WINNER in ARCH_T2
run(f"python train.py --architecture {WINNER} --data_dir {DATA_DIR} {ARCH_T2[WINNER]} --context_length {T2_CTX} "
    f"--max_train_tokens {S['t2_tokens']} --max_steps {S['t2_steps']} --warmup_steps {S['t2_warm']} "
    f"--batch_size 8 --grad_accum_steps 4 --lr 3e-4 --eval_every {S['t2_eval']} --log_every 50 "
    f"--eval_tokens {S['eval_tokens']} --early_stopping_patience {S['t2_patience']} --early_stopping_min_delta 0.002 "
    f"--time_budget_min {S['t2_budget']} --checkpoint_dir artifacts/ckpt_tier2 --tag t2_{WINNER}")
show("train", "t2_", TRAIN_COLS)

## 11. Checkpoint selection: best vs. final (§7.4)

Does the lowest-val-loss checkpoint actually generate better than the last-step one? (If training ended at its best step the two are identical — say so in the report.)

In [ ]:
for which in ("best", "final"):
    run(f"python eval.py --checkpoint artifacts/ckpt_tier2/{WINNER}_{which}.pt --tokenizer_path artifacts/tokenizer.json "
        f"--data_dir {DATA_DIR} --n_examples {S['n_eval']} --tag t2_{which}")
show("eval", "t2_", EVAL_COLS)

## 12. Decoding sweep (§7.4, §7.8)

In [ ]:
DEC_CONFIGS = [
    dict(temperature=0.7, top_p=0.9,  top_k=40, repetition_penalty=1.1),
    dict(temperature=0.8, top_p=0.9,  top_k=50, repetition_penalty=1.2),
    dict(temperature=1.0, top_p=0.95, top_k=0,  repetition_penalty=1.0),
]
for i, c in enumerate(DEC_CONFIGS):
    run(f"python eval.py --checkpoint artifacts/ckpt_tier2/{WINNER}_best.pt --tokenizer_path artifacts/tokenizer.json "
        f"--data_dir {DATA_DIR} --n_examples {S['n_eval']} --temperature {c['temperature']} --top_p {c['top_p']} "
        f"--top_k {c['top_k']} --repetition_penalty {c['repetition_penalty']} --tag sweep_{i}")
show("eval", "sweep_", EVAL_COLS)

## 13. Model compression via distillation (§7.4)

Trains a smaller student (384 hidden / 6 layers ≈ 22M params) against the Tier-2 model's soft outputs. It is an experimental artifact for the size-vs-quality analysis (§7.4/§7.9); it doesn't need to be in the 50–300M range. (Not available for an `enc_dec` winner.)

In [ ]:
if WINNER == "enc_dec":
    print("distill.py does not support enc_dec teachers -- skipping (say so in the report, or distil a decoder-style Tier-1 model).")
else:
    run(f"python distill.py --teacher_checkpoint artifacts/ckpt_tier2/{WINNER}_best.pt --data_dir {DATA_DIR} "
        f"--student_hidden_size 384 --student_num_layers 6 --student_num_heads 6 --student_num_kv_heads 3 "
        f"--max_train_tokens {S['dist_tokens']} --max_steps {S['dist_steps']} --eval_every {S['dist_eval']} "
        f"--eval_tokens {S['eval_tokens']} --tag distilled_student")
    run(f"python eval.py --checkpoint artifacts/ckpt_distill/{WINNER}_student_distilled.pt --tokenizer_path artifacts/tokenizer.json "
        f"--data_dir {DATA_DIR} --n_examples {S['n_eval']} --tag distilled_eval")
    show("eval", "distilled", EVAL_COLS)

## 14. Official predictions + Codabench zip (§7.2.5)

The decoding config is picked automatically from the sweep (highest BERTScore F1, falling back to ROUGE-L). Generation is resumable (`--resume`) and writes partial results every 25 prompts.

In [ ]:
RESUME_GENERATION = False      # True: continue a partially written artifacts/prediction.json after a timeout
df = load_results()
sw = df[(df.kind == "eval") & df.tag.str.startswith("sweep_")]
metric = "bertscore_f1" if sw.bertscore_f1.notna().any() else "rougeL_f1"
best = sw.sort_values(metric, ascending=False).iloc[0]
DEC = dict(temperature=float(best.temperature), top_p=float(best.top_p), top_k=int(best.top_k),
           repetition_penalty=float(best.repetition_penalty))
print(f"decoding chosen by {metric}: {DEC}")
if os.path.exists("artifacts/prediction.json") and not RESUME_GENERATION:
    os.remove("artifacts/prediction.json")
run(f"python generate.py --checkpoint artifacts/ckpt_tier2/{WINNER}_best.pt --tokenizer_path artifacts/tokenizer.json "
    f"--test_file {TEST_FILE} --out artifacts/prediction.json --zip_out artifacts/submission.zip "
    f"--temperature {DEC['temperature']} --top_p {DEC['top_p']} --top_k {DEC['top_k']} "
    f"--repetition_penalty {DEC['repetition_penalty']} {'--resume' if RESUME_GENERATION else ''}")

In [ ]:
# Sanity check the output format against spec 7.2.5
import zipfile
with open("artifacts/prediction.json") as f:
    preds = json.load(f)
test = json.load(open(TEST_FILE))
assert set(preds) == set(test), f"keys differ: {len(set(test) - set(preds))} missing, {len(set(preds) - set(test))} extra"
assert all(isinstance(v, dict) and set(v) == {"text"} for v in preds.values()), "each entry must be {'text': ...}"
nw = pd.Series([len(v["text"].split()) for v in preds.values()])
print(f"n predictions: {len(preds)} | words per prediction: min {nw.min()}, median {nw.median():.0f}, max {nw.max()} (limit 128)")
print("empty predictions:", int((nw == 0).sum()))
assert nw.max() <= 128
with zipfile.ZipFile("artifacts/submission.zip") as z:
    print("submission.zip contains:", z.namelist())
for k in list(preds)[:3]:
    print(f"--- {k} ---\nPROMPT (tail): ...{test[k]['text'][-200:]}\nCONTINUATION: {preds[k]['text']}\n")

## 15. Everything in one table (feeds the §7.9 report)

In [ ]:
print("TRAINING RUNS"); show("train", None, TRAIN_COLS + ["teacher_params_M", "teacher_val_loss"])
print("GENERATION EVALS"); show("eval", None, EVAL_COLS)
load_results().to_csv("artifacts/results_summary.csv", index=False)
print(subprocess.run("du -sh artifacts/* | sort -h | tail -12", shell=True, capture_output=True, text=True).stdout)
print("Download: artifacts/submission.zip (Codabench), artifacts/results_summary.csv (report tables), artifacts/ckpt_tier2/*.pt (model)")

## 16. Report checklist (§7.9)

Every item below has an executed cell above with real numbers (`artifacts/results_summary.csv`):

- [ ] Tier-1 val loss/ppl + param count for all 4 architectures (§7), and why each counts as "substantially different" (dense GQA decoder / encoder-decoder / local+global attention / chunk-recurrent memory). Remember val_loss isn't comparable for `enc_dec`.
- [ ] Tokenizer vocab-size comparison, tokens/word overall and per source (§4)
- [ ] Data-mixing ablation: uniform vs judgment-heavy (§8)
- [ ] Context-length ablation: 512 vs 1024 — val_loss *and* BERTScore/ROUGE-L (§9)
- [ ] Regularization ablation: default vs heavier dropout/weight decay, and why the gap is small at this scale (§9b)
- [ ] Best-vs-final checkpoint comparison (§11)
- [ ] Decoding sweep + the config picked (§12)
- [ ] Distillation: student size vs teacher size vs quality drop (§13)
- [ ] Relationship (or mismatch) between validation perplexity and BERTScore / ROUGE-L / BLEU-4 across all runs
- [ ] Which model(s) you carry into Phase 2
- [ ] Bugs found and fixed vs. the first attempt (list in the header of this notebook) — worth a short "engineering notes" paragraph